# W04_5 사진을 치수로 읽기 — 건물 기술서(BDS)와 결정론적 3D

<a href="https://colab.research.google.com/github/kloud80/dataminig_cau/blob/main/%EC%A4%91%EC%95%99%EB%8C%8026%ED%8A%B9%EA%B0%95/notebooks/w04_colab/W04_5%20%EC%82%AC%EC%A7%84%EC%9D%84%20%EC%B9%98%EC%88%98%EB%A1%9C%20%EC%9D%BD%EA%B8%B0%20%E2%80%94%20BDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **LLM 은 계측기이지 생성기가 아니다.** 모델은 숫자(JSON)만 내놓고, 그 숫자로 삼각형을 찍는 것은 전부 일반 코드다.
> 같은 숫자를 넣으면 항상 같은 메시가 나온다.

사진 한 장으로 "창이 1.5 m 인지 1.8 m 인지" 맞힐 수 있을까? 사람도 못 한다 — 사진에는 **축척이 없다.**
그런데 "창 폭이 기둥 간격의 60 %", "창 높이가 층고의 절반" 은 사람도 모델도 잘 본다.
그리고 우리는 이미 **입면 폭·건물 높이·층수**를 공공데이터로 알고 있다. 그래서 `addr2blend` v0.3 은

1. 모델에게 **비율만** 묻고 (미터는 묻지 않는다)
2. 비율 × 알려진 치수 = **미터**로 바꿔 건물 기술서(BDS, Building Description Schema)에 적고
3. 그 기술서를 **결정론적 코드**로 3D 로 만든다.

| 단계 | 이 노트북 | addr2blend 원본 | 하는 일 |
|---|---|---|---|
| 1 | 알려진 치수 | `s04_geometry` | 입면 폭·높이·층수 — 공공데이터 |
| 2 | 촬영 계획 | `s21_survey` (LLM ①) | 어느 입면을 어떤 구도로 몇 장 찍을지 — 여기서는 계획만 본다 |
| 3 | 비율 판독 | `s22_reading` · `bds/reader.py` (LLM ②) | 사진 → **비율 JSON** (strict 스키마) |
| 4 | 미터 환산 | `apply_facade_reading` | 비율 × 알려진 치수 → BDS |
| 5 | 결정론적 생성 | `s20_procedural` · `blender/build_model.py` | BDS → 벽·창·간판 메시 → `.glb` |

**예제 건물** 아이파크타워 (서울 강남구 영동대로 520). 4번 노트북을 먼저 돌렸으면 그 옥상 판독도 함께 올린다.

> **키가 없어도 끝까지 돈다.** `OPENAI_API_KEY` 가 없으면 실제 파이프라인(gpt-5.5)이 이 건물에서 읽은 결과(`s22_bds.json`)를 쓴다.

## 1. 준비

In [ ]:
import os, sys, io, json, math, time, base64, subprocess, importlib
from pathlib import Path

# ── 데이터 찾기 ─────────────────────────────────────────────
REPO_URL = "https://github.com/kloud80/dataminig_cau.git"  # 강의 저장소
REPO_DIR = "중앙대26특강/notebooks/w04_colab"   # 저장소 안에서 이 노트북들이 있는 폴더

def find_base():
    for b in [Path.cwd(), Path.cwd() / REPO_DIR, Path("/content") / REPO_DIR, Path("/content/repo") / REPO_DIR]:
        if (b / "data" / "ipark" / "s04_geometry.json").exists():
            return b
    return None

BASE = find_base()
if BASE is None and REPO_URL:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO_URL, "/content/repo"], check=False)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", REPO_DIR], check=False)
    BASE = find_base()
if BASE is None:
    raise SystemExit(
        "data/ipark 폴더를 찾지 못했습니다.\n"
        "  (1) 네트워크를 확인하고 이 셀을 다시 실행 (강의 저장소에서 자동으로 받는다)\n"
        "  (2) 또는 왼쪽 파일 탭에 w04_colab 폴더(data/ipark 포함)를 통째로 올린 뒤 다시 실행")
DATA = BASE / "data" / "ipark"
OUT = BASE / "out"
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(BASE))          # w04_tools.py (보기용 도우미)

# ── 패키지 (Colab 에 없는 것만 설치) ────────────────────────
def need(mod, pip_name=None):
    try:
        return importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or mod], check=True)
        return importlib.import_module(mod)

import copy, re
import numpy as np
import requests
from PIL import Image
need("shapely")
need("mapbox_earcut", "mapbox-earcut")
trimesh = need("trimesh")
from shapely.geometry import Polygon
from w04_tools import PartSet, box_vf, render, show_glb, sha, hex_rgb

# ── API 키: Colab 보안 비밀 → 환경변수. 값은 절대 출력하지 않는다 ──
def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    v = os.environ.get(name, "").strip()
    return v or None

OPENAI_KEY = get_key("OPENAI_API_KEY")
print("OPENAI_API_KEY:", "있음" if OPENAI_KEY else "없음 → 실제 파이프라인의 판독 결과(s22_bds.json) 사용")
print("데이터:", DATA)
print("결과물:", OUT)

In [ ]:
# 한글 폰트 — 그래프 글자가 네모로 깨지지 않게 (Colab 은 나눔고딕을 설치, 윈도는 맑은 고딕)
import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
FONT_PATH = None
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgunbd.ttf'
else:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf'
    if not os.path.exists(nanum):
        os.system('apt-get -qq install -y fonts-nanum > /dev/null')
    if os.path.exists(nanum):
        fm.fontManager.addfont(nanum)
        fm.fontManager.addfont(nanum.replace('Bold', ''))
        plt.rcParams['font.family'] = 'NanumGothic'
        FONT_PATH = nanum
plt.rcParams['axes.unicode_minus'] = False

## 2. 이미 아는 것 — 입면의 폭, 건물의 높이, 층수

`s04_geometry.json` 은 외곽선의 변 하나하나를 **입면**으로 나눠 둔 것이다. 입면마다 폭(m), 바깥 방향(법선), 길가인지가 있다.
여기에 **사진을 읽을 필요가 없는** 사실이 다 들어 있다.

In [ ]:
geom = json.loads((DATA / "s04_geometry.json").read_text(encoding="utf-8"))
S22 = json.loads((DATA / "s22_bds.json").read_text(encoding="utf-8"))
FP = np.array(geom["footprint"])
H = float(geom["height_m"])
N_FLOORS = len(geom["floors"])
GEO = {f["id"]: f for f in geom["facades"]}

def compass(normal):
    az = (math.degrees(math.atan2(normal[0], normal[1])) + 360) % 360
    names = ["north", "north-east", "east", "south-east", "south", "south-west", "west", "north-west"]
    return names[int((az + 22.5) // 45) % 8], az

print(f"높이 {H:.2f} m · 지상 {N_FLOORS}층 · 층고(대장 기준 평균) {H / N_FLOORS:.2f} m\n")
print(f"{'입면':5s} {'폭(m)':>7s}  {'바라보는 쪽':16s} 길가")
for f in geom["facades"]:
    c, az = compass(f["normal"])
    print(f"{f['id']:5s} {f['width_m']:7.1f}  {c:11s}({az:3.0f}°) {'예' if f['street_facing'] else ''}")

## 3. 실제 파이프라인은 사진을 어떻게 모았나 — 촬영 계획만 본다

`s21_survey` 단계에서 모델(LLM ①)이 "어느 입면이 중요하고, 몇 장을, 어떤 구도로" 찍을지 계획을 세우고,
코드가 그 계획대로 거리뷰 카메라의 위치·방향·화각을 역산해 22장을 찍었다.

**그 사진들은 내부 연구용이라 이 수업에서는 쓰지 않는다.** 여기서는 카메라 파라미터만 지도에 찍어 본다.
대신 4절에서 공개 라이선스 사진(또는 직접 찍은 사진)을 쓴다.

In [ ]:
plan = json.loads((DATA / "s21_survey_plan.json").read_text(encoding="utf-8"))
shots = json.loads((DATA / "s21_survey_shots.json").read_text(encoding="utf-8"))["shots"]
for t in plan["targets"][:6]:
    print(f"{t['facade_id']:5s} 중요도 {t['importance']:2d} · {t['shots']}장 {t['framings']}")

COLS = {"full": "#007aff", "ground": "#ff9500", "upper": "#34c759", "corner": "#af52de"}
fig, ax = plt.subplots(figsize=(8, 8))
ax.fill(*FP.T, color="#dfe6ee", ec="#33475b", lw=2)
for f in geom["facades"]:
    if f["width_m"] > 9:
        (sx, sy), (ex, ey) = f["start"], f["end"]
        ax.text((sx + ex) / 2 + f["normal"][0] * 5, (sy + ey) / 2 + f["normal"][1] * 5, f["id"], ha="center", fontsize=11)
for s in shots:
    x, y, _ = s["position"]
    a = math.radians(s["pan_deg"] + s["heading_correction_deg"])   # 진북 기준 방위
    ax.annotate("", (x + 9 * math.sin(a), y + 9 * math.cos(a)), (x, y),
                arrowprops=dict(arrowstyle="->", color=COLS[s["framing"]], lw=1.8))
    ax.plot(x, y, "o", color=COLS[s["framing"]], ms=5)
for k, c in COLS.items():
    ax.plot([], [], "o-", color=c, label=k)
ax.legend(title="구도", loc="upper right"); ax.set_aspect("equal"); ax.grid(alpha=.3)
ax.set_title(f"실제 run 의 촬영 계획 — 카메라 {len(shots)}대 (위치·방향만)", fontsize=14); plt.show()

## 4. 사진 고르기

* **직접 찍은 사진** — 왼쪽 파일 탭에 올리고 `PHOTO_PATH` 에 적는다. 그리고 `FACADE_ID` 를 2절 표에서 사진 속 입면으로 고른다.
  다른 건물을 찍었다면 아래 `KNOWN` 의 폭·높이·층수를 그 건물 값으로 바꾼다 (건축물대장, 지도 거리재기).
* **비워 두면** Wikimedia Commons 의 공개 라이선스 사진을 쓴다 — 영동대로 건너편에서 본 아이파크타워의 긴 서남서 면(`f_0`).
  라이선스는 Commons API 로 그 자리에서 확인하고, 저작자 표시를 그림에 단다 (CC BY-SA 4.0 은 저작자 표시가 의무).

In [ ]:
PHOTO_PATH = ""                 # 예) "my_building.jpg"
FACADE_ID = "f_0"
PHOTO_NOTE = "The photo is taken from across the road at an oblique angle; the facade recedes to the right. Measure only this facade."

COMMONS_TITLE = "File:Yeongdong-daero 3.jpg"
OPEN_LICENCES = ("CC BY", "CC0", "Public domain", "PD")

def strip_html(s):
    return re.sub(r"<[^>]+>", "", s or "").strip()

def commons_photo(title, width=1280):
    """Commons API 로 라이선스를 확인하고 사진을 받는다. 공개 라이선스가 아니면 거부."""
    S = requests.Session()
    S.headers["User-Agent"] = "urban-datamining-course/1.0 (W04 notebook)"
    r = S.get("https://commons.wikimedia.org/w/api.php", timeout=15, params=dict(
        action="query", titles=title, prop="imageinfo", iiprop="url|extmetadata", iiurlwidth=width, format="json"))
    ii = next(iter(r.json()["query"]["pages"].values()))["imageinfo"][0]
    m = ii["extmetadata"]
    lic = m["LicenseShortName"]["value"]
    if not lic.startswith(OPEN_LICENCES):
        raise ValueError("공개 라이선스가 아니다: " + lic)
    img = Image.open(io.BytesIO(S.get(ii["thumburl"], timeout=30).content)).convert("RGB")
    return img, {"title": title, "author": strip_html(m["Artist"]["value"]), "licence": lic,
                 "page": ii["descriptionurl"], "checked": "Commons API (실시간)"}

if PHOTO_PATH:
    PHOTO = Image.open(PHOTO_PATH).convert("RGB")
    CREDIT = {"title": PHOTO_PATH, "author": "수강생 촬영", "licence": "-", "checked": "-"}
else:
    try:
        PHOTO, CREDIT = commons_photo(COMMONS_TITLE)
    except Exception as e:
        print("Commons 접속 실패 → 저장본 사용:", type(e).__name__)
        PHOTO = Image.open(BASE / "data" / "commons" / "Yeongdong-daero_3.jpg").convert("RGB")
        a = json.loads((BASE / "data" / "commons" / "ATTRIBUTION.json").read_text(encoding="utf-8"))
        CREDIT = {"title": a["title"], "author": a["author"], "licence": a["licence"], "page": a["page"],
                  "checked": a["verified_via"]}
credit_line = f"사진: {CREDIT['author']} · {CREDIT['licence']} · {CREDIT.get('page', '')}"
print(credit_line, "\n라이선스 확인:", CREDIT["checked"])

geo = GEO[FACADE_ID]
KNOWN = {"width_m": geo["width_m"], "height_m": H, "floors": N_FLOORS,
         "facade_id": FACADE_ID, "compass": compass(geo["normal"])[0]}
print("알려진 치수:", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in KNOWN.items()})

plt.figure(figsize=(12, 8.5)); plt.imshow(PHOTO); plt.axis("off")
plt.title(f"입면 {FACADE_ID} — 폭 {KNOWN['width_m']:.1f} m, 높이 {H:.1f} m, {N_FLOORS}층", fontsize=15)
plt.figtext(0.5, 0.06, credit_line, ha="center", fontsize=10, color="#555"); plt.show()

## 5. 비율로 묻는 스키마

원본 `bds/reader.py` 스키마에서 핵심만 남겼다. **미터 단위 칸이 하나도 없다.**

| 칸 | 뜻 | 나중에 곱하는 것 |
|---|---|---|
| `floor_line_fracs` | 층 경계의 높이 비율 (0 바닥 → 1 옥상) | × 건물 높이 |
| `bay_count` | 기둥 간격(베이) 수 — 창 수가 아니라 **반복 리듬** | 입면 폭 ÷ 베이 수 = 베이 폭 |
| `window.w_frac_of_bay` | 창 폭 ÷ 베이 폭 | × 베이 폭 |
| `window.h_frac_of_floor`, `sill_…`, `recess_…` | 창 높이·창대·안으로 들어간 깊이 ÷ 층고 | × 층고 |
| `ground.use`, `units[].x0_norm…` | 1층 용도, 점포 구간 (입면 폭 비율) | × 입면 폭 |
| `signs[]` | 간판 종류·**글자 그대로**·위치 비율·색 | × 폭, × 높이 |
| `need_shots` | 이 사진으로 못 읽은 것 → **어떤 구도를 더 찍어야 하나** | 다음 촬영 계획 |
| `confidence` | 전체 신뢰도 0~1 | 0.25 미만이면 버린다 |

모르면 지어내지 말고 `need_shots` 로 요청하라고 시킨다. 이것이 1번 촬영 계획으로 되돌아가는 고리다.

In [ ]:
MATERIALS = ["concrete", "paint", "tile", "brick", "stone", "metal_panel", "curtain_wall", "wood", "glass_block", "other"]
GROUND_USES = ["storefront", "lobby", "office", "residential", "parking", "wall", "unknown"]
WINDOW_KINDS = ["punched", "ribbon", "curtain_wall", "none"]
SIGN_KINDS = ["board", "vertical", "projecting", "channel_letters", "window_text"]
FRAMINGS = ["full", "ground", "upper", "corner"]

def obj(props, req=None):
    return {"type": "object", "additionalProperties": False, "required": req or list(props), "properties": props}
NUM, INT, STR, BOOL = {"type": "number"}, {"type": "integer"}, {"type": "string"}, {"type": "boolean"}
enum = lambda xs: {"type": "string", "enum": xs}
arr = lambda x: {"type": "array", "items": x}

SCHEMA = obj({
    "floors_visible": INT,
    "top_visible": BOOL,
    "floor_line_fracs": arr(NUM),
    "wall_material": enum(MATERIALS),
    "wall_color_hex": STR,
    "bay_count": INT,
    "window": obj({"kind": enum(WINDOW_KINDS), "per_bay": INT,
                   "w_frac_of_bay": NUM, "h_frac_of_floor": NUM, "sill_frac_of_floor": NUM,
                   "recess_frac_of_floor": NUM, "cols_per_window": INT, "rows_per_window": INT,
                   "frame_color_hex": STR, "glass_color_hex": STR}),
    "ground": obj({"use": enum(GROUND_USES),
                   "units": arr(obj({"x0_norm": NUM, "x1_norm": NUM,
                                     "door_x_norm": {"anyOf": [NUM, {"type": "null"}]},
                                     "bulkhead_frac": NUM, "header_frac": NUM,
                                     "has_awning": BOOL, "awning_color_hex": STR}))}),
    "signs": arr(obj({"kind": enum(SIGN_KINDS), "text": STR, "x0_norm": NUM, "x1_norm": NUM,
                      "z_frac": NUM, "h_frac": NUM, "board_color_hex": STR, "text_color_hex": STR})),
    "need_shots": arr(obj({"framing": enum(FRAMINGS), "why": STR})),
    "confidence": NUM,
    "notes": STR,
})

PROMPT = """You are measuring ONE facade of a real building from street-level photos, so it can be
rebuilt as 3D geometry. Report PROPORTIONS, never absolute metres — the true dimensions are already
known and will be multiplied in.

Known facts (do not contradict them):
- This facade is {width_m:.1f} m wide and the building is {height_m:.1f} m tall.
- The building has {floors} floors above ground (from the official register).
- Facade id {facade_id}, facing {compass}.
- {photo_note}

Coordinate conventions:
- x_norm / x0_norm / x1_norm: 0 = LEFT edge of this facade, 1 = RIGHT edge, as seen from the street.
- z_frac: 0 = ground, 1 = top of the building.
- floor_line_fracs: the height fraction of each floor boundary, ascending, excluding 0 and including
  the roofline if visible. For 5 equal floors that would be about [0.2, 0.4, 0.6, 0.8, 1.0].
- *_frac_of_floor: divided by ONE typical floor height, so a 1.5 m window in a 3 m floor is 0.5.
- w_frac_of_bay: window width divided by the width of one structural bay.

Rules:
- Count bays by the repeating vertical rhythm of the facade, not by the number of windows.
- Colours: read the actual dominant colour of the surface as a hex string, in daylight terms.
- Signs: transcribe text exactly as written (Korean or Latin), or "" if unreadable. Ignore signs on other buildings.
- If a part of this facade is not readable from the given photos, do NOT guess: add an entry to
  need_shots saying which framing would fix it. Keep need_shots empty if the photos suffice.
- confidence 0..1 for the whole reading."""
print(f"스키마 최상위 칸 {len(SCHEMA['properties'])}개:", ", ".join(SCHEMA["properties"]))

## 6. 판독 — 모델에게 묻거나, 실제 run 의 결과를 불러오거나

* 키가 있으면: Responses API + `json_schema` + `strict: True`. 모델 이름은 변수다 (파이프라인 기본값 `gpt-5.5`).
* 키가 없으면: 실제 run 이 저장한 `s22_bds.json` 의 이 입면은 **이미 미터로 환산된** 값이다. 그것을 같은 스키마의
  **비율로 되돌려** 판독 결과 자리에 놓는다 — 그다음 과정(7절~)은 두 경우가 똑같다.

In [ ]:
def to_data_url(img, max_side=1024, fmt="PNG"):
    """numpy 배열·PIL 이미지 → data URL (긴 변 max_side 이하로 줄여서)."""
    im = Image.fromarray(img) if isinstance(img, np.ndarray) else img.convert("RGB")
    s = max_side / max(im.size)
    if s < 1:
        im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, fmt, **({"quality": 90} if fmt == "JPEG" else {}))
    return f"data:image/{fmt.lower()};base64," + base64.b64encode(buf.getvalue()).decode()

VLM_MODEL = "gpt-5.5"
REASONING = "low"

def reading_from_bds(fb, W, Hb, fh):
    """s22_bds 의 미터 값 → 판독 스키마의 비율 (실제 run 의 판독을 되살린다)."""
    pat = lambda k: next((p for p in fb["patterns"] if k in p["floors"]), fb["patterns"][-1])
    up, p1 = pat(min(2, len(fh))), pat(1)
    win, typ = up["window"], float(np.median(fh))
    bay_n = fb.get("bay_count") or len(fb["bays"])
    bay_w = W / bay_n
    return {
        "floors_visible": len(fh), "top_visible": True,
        "floor_line_fracs": [round(float(t) / Hb, 4) for t in np.cumsum(fh)],
        "wall_material": fb["wall"]["class"] if fb["wall"]["class"] in MATERIALS else "other",
        "wall_color_hex": fb["wall"]["color"], "bay_count": bay_n,
        "window": {"kind": win["kind"], "per_bay": win["per_bay"],
                   "w_frac_of_bay": round(win["w_m"] * win["per_bay"] / bay_w, 3),
                   "h_frac_of_floor": round(win["h_m"] / typ, 3), "sill_frac_of_floor": round(win["sill_m"] / typ, 3),
                   "recess_frac_of_floor": round(win["recess_m"] / typ, 4),
                   "cols_per_window": win["mullions"][0], "rows_per_window": win["mullions"][1],
                   "frame_color_hex": win["frame_color"], "glass_color_hex": win["glass_color"]},
        "ground": {"use": p1["use"], "units": [
            {"x0_norm": u["x0_m"] / W, "x1_norm": u["x1_m"] / W,
             "door_x_norm": None if u.get("door_x_m") is None else u["door_x_m"] / W,
             "bulkhead_frac": u["bulkhead_m"] / fh[0], "header_frac": u["header_m"] / fh[0],
             "has_awning": bool(u.get("awning")), "awning_color_hex": (u.get("awning") or {}).get("color", "#7E7468")}
            for u in fb.get("storefront", [])]},
        "signs": [{"kind": s["kind"], "text": s["text"], "x0_norm": round(s["x0_m"] / W, 3), "x1_norm": round(s["x1_m"] / W, 3),
                   "z_frac": round(s["z_m"] / Hb, 3), "h_frac": round(s["h_m"] / Hb, 3),
                   "board_color_hex": s["board_color"], "text_color_hex": s["text_color"]} for s in fb.get("signs", [])],
        "need_shots": [], "confidence": fb["ev"]["confidence"],
        "notes": "s22_bds.json 의 미터 값을 비율로 되돌린 것 (실제 run: gpt-5.5, 거리뷰 판독)"}

READING, READ_META = None, {}
if OPENAI_KEY:
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_KEY, timeout=240, max_retries=1)
        prompt = PROMPT.format(photo_note=PHOTO_NOTE, **KNOWN)
        kwargs = dict(model=VLM_MODEL, input=[{"role": "user", "content": [
                          {"type": "input_text", "text": prompt},
                          {"type": "input_image", "image_url": to_data_url(PHOTO, 1400, "JPEG")}]}],
                      text={"format": {"type": "json_schema", "name": "facade_bds", "schema": SCHEMA, "strict": True}})
        if REASONING:
            kwargs["reasoning"] = {"effort": REASONING}
        t0 = time.time()
        resp = client.responses.create(**kwargs)
        READING = json.loads(resp.output_text)
        READ_META = {"source": f"openai:{VLM_MODEL}", "latency_s": round(time.time() - t0, 1),
                     "input_tokens": resp.usage.input_tokens, "output_tokens": resp.usage.output_tokens}
        print(f"{VLM_MODEL} 판독 {READ_META['latency_s']}초 · 토큰 입력 {resp.usage.input_tokens} / 출력 {resp.usage.output_tokens}")
    except Exception as e:
        print("판독 호출 실패 →", type(e).__name__, str(e)[:300])
if READING is None:
    if FACADE_ID not in S22["facades"]:
        raise SystemExit(f"{FACADE_ID} 는 실제 run 에 판독 결과가 없다. FACADE_ID 를 f_0 / f_1 / f_7 처럼 바꿀 것")
    READING = reading_from_bds(S22["facades"][FACADE_ID], GEO[FACADE_ID]["width_m"], H, S22["floor_heights_m"])
    READ_META = {"source": "staged:s22_bds.json (실제 run, gpt-5.5)"}
    print("API 판독 없이 진행 — 실제 run 의 판독을 비율로 되돌려 쓴다")
print(json.dumps(READING, ensure_ascii=False, indent=1)[:2200])

## 7. 비율 × 알려진 치수 = 미터

원본 `apply_facade_reading` 을 줄여 옮겼다. 하는 일은 곱셈과 **조이기**(clip)뿐이다.

* 층 경계 개수가 대장 층수와 같고 층고가 1.8~8 m 사이일 때만 층고를 바꾼다 — 대장 층수는 사진보다 믿는다.
* 베이 수는 1.2 m 보다 좁아지지 않게, 창 폭은 베이의 10~95 % 로, 창 높이는 층고의 15~90 % 로 조인다.
* 모델이 터무니없는 수를 내도 **건물이 망가지지 않는 범위** 안에 갇힌다.

In [ ]:
def _clip(v, lo, hi, default):
    try:
        return float(np.clip(float(v), lo, hi))
    except (TypeError, ValueError):
        return default

def _hex(v, default):
    v = (v or "").strip()
    return v if v.startswith("#") and len(v) in (4, 7) else default

def apply_reading(r, W, Hb, n, fh_old):
    """판독(비율) → (입면 BDS(미터), 층고 목록, 메모)."""
    notes = []
    fh = list(fh_old)
    lines = sorted(float(x) for x in r.get("floor_line_fracs", []) if 0.02 < float(x) <= 1.0)
    if len(lines) == n:
        h = np.diff(np.concatenate([[0.0], np.array(lines) * Hb]))
        if np.all(h > 1.8) and np.all(h < 8.0):
            fh = [round(float(x), 3) for x in h]
            notes.append(f"층 경계 {len(lines)}개 → 층고 {fh[0]:.2f}, {fh[1]:.2f}, … m")
    else:
        notes.append(f"층 경계 {len(lines)}개 ≠ 대장 {n}층 → 층고는 그대로")
    typ = float(np.median(fh))
    bay_n = int(_clip(r.get("bay_count"), 1, max(1, int(W / 1.2)), max(1, round(W / 3.2))))
    bay_w = W / bay_n
    w = r.get("window") or {}
    per_bay = int(_clip(w.get("per_bay"), 1, 4, 1))
    win = {"kind": w.get("kind") if w.get("kind") in WINDOW_KINDS else "punched", "per_bay": per_bay,
           "w_m": round(_clip(w.get("w_frac_of_bay"), 0.1, 0.95, 0.5) * bay_w / per_bay, 2),
           "h_m": round(_clip(w.get("h_frac_of_floor"), 0.15, 0.9, 0.5) * typ, 2),
           "sill_m": round(_clip(w.get("sill_frac_of_floor"), 0.0, 0.7, 0.3) * typ, 2),
           "recess_m": round(_clip(w.get("recess_frac_of_floor"), 0.0, 0.15, 0.035) * typ, 3),
           "mullions": [int(_clip(w.get("cols_per_window"), 1, 6, 2)), int(_clip(w.get("rows_per_window"), 1, 6, 1))],
           "frame_color": _hex(w.get("frame_color_hex"), "#C8CCD0"), "glass_color": _hex(w.get("glass_color_hex"), "#8FA9BC"),
           "frame_w_m": 0.06}
    g = r.get("ground") or {}
    guse = g.get("use") if g.get("use") in GROUND_USES else "wall"
    store = []
    if guse == "storefront":
        for u in g.get("units", [])[:8]:
            x0, x1 = _clip(u.get("x0_norm"), 0, 1, 0) * W, _clip(u.get("x1_norm"), 0, 1, 1) * W
            if x1 - x0 < 1.0:
                continue
            dx = u.get("door_x_norm")
            store.append({"x0_m": round(x0, 2), "x1_m": round(x1, 2),
                          "door_x_m": None if dx is None else round(_clip(dx, 0, 1, .5) * W, 2),
                          "bulkhead_m": round(_clip(u.get("bulkhead_frac"), 0, .45, .15) * fh[0], 2),
                          "header_m": round(_clip(u.get("header_frac"), .05, .45, .14) * fh[0], 2),
                          "awning": {"color": _hex(u.get("awning_color_hex"), "#7E7468"), "depth_m": 0.9} if u.get("has_awning") else None})
        if not store:
            guse = "lobby"
    signs = []
    for s in r.get("signs", [])[:6]:
        x0, x1 = _clip(s.get("x0_norm"), 0, 1, 0) * W, _clip(s.get("x1_norm"), 0, 1, 1) * W
        if x1 - x0 < 0.3 and s.get("kind") not in ("vertical", "projecting"):
            continue
        signs.append({"kind": s.get("kind") if s.get("kind") in SIGN_KINDS else "board", "text": (s.get("text") or "")[:40],
                      "x0_m": round(x0, 2), "x1_m": round(max(x1, x0 + 0.3), 2),
                      "z_m": round(_clip(s.get("z_frac"), 0, 1, .3) * Hb, 2),
                      "h_m": round(max(0.25, _clip(s.get("h_frac"), .01, .25, .05) * Hb), 2),
                      "board_color": _hex(s.get("board_color_hex"), "#1A1A1A"), "text_color": _hex(s.get("text_color_hex"), "#F2F2F0")})
    wall = {"class": r.get("wall_material", "other"), "color": _hex(r.get("wall_color_hex"), "#C9C4BA")}
    patterns = [{"floors": [1], "use": guse, "window": win if guse != "storefront" else {**win, "kind": "none"}}]
    if n > 1:
        patterns.append({"floors": list(range(2, n + 1)), "use": "residential" if guse == "residential" else "office", "window": win})
    fb = {"id": KNOWN["facade_id"], "role": "front", "wall": wall, "bay_count": bay_n,
          "bays": [[i * bay_w, (i + 1) * bay_w] for i in range(bay_n)], "patterns": patterns,
          "storefront": store, "signs": signs, "ev": {"confidence": float(r.get("confidence", 0))}}
    notes.append(f"베이 {bay_n}개 × {bay_w:.2f} m, 창 {win['kind']} {win['w_m']:.2f} × {win['h_m']:.2f} m, "
                 f"창대 {win['sill_m']:.2f} m, 리빌 {win['recess_m'] * 100:.0f} cm, 1층 {guse}, 간판 {len(signs)}")
    return fb, fh, notes

if float(READING.get("confidence", 0)) < 0.25:
    raise SystemExit("판독 신뢰도가 0.25 미만 — 기본값을 유지한다 (사진을 바꿔 다시)")
FACADE_M, FH, notes = apply_reading(READING, KNOWN["width_m"], H, N_FLOORS, S22["floor_heights_m"])
for t in notes:
    print("·", t)

w = READING["window"]; bay_w = KNOWN["width_m"] / FACADE_M["bay_count"]; typ = float(np.median(FH))
print(f"\n{'항목':10s} {'판독(비율)':>10s}   ×  {'알려진 치수':>14s}   =  {'미터':>7s}")
for k, frac, base, lab in [("베이 폭", 1 / FACADE_M["bay_count"], KNOWN["width_m"], "입면 폭"),
                           ("창 폭", w["w_frac_of_bay"], bay_w, "베이 폭"),
                           ("창 높이", w["h_frac_of_floor"], typ, "층고"),
                           ("창대", w["sill_frac_of_floor"], typ, "층고"),
                           ("리빌 깊이", w["recess_frac_of_floor"], typ, "층고")]:
    print(f"{k:10s} {frac:10.3f}   ×  {lab:>6s} {base:6.2f} m   =  {frac * base:6.2f} m")

if READING.get("need_shots"):
    print("\n추가 촬영 요청:")
    for s in READING["need_shots"]:
        print(f"  [{s['framing']}] {s['why']}")

### 같은 비율, 다른 건물

비율은 **축척과 무관하다.** 같은 판독을 폭 35 m 짜리 입면에 얹으면 창도 절반이 된다.
그래서 판독의 정확도는 사진이 아니라 **공공데이터의 치수**가 받친다.

In [ ]:
for W_try in (KNOWN["width_m"], KNOWN["width_m"] / 2):
    fb_try, _, _ = apply_reading(READING, W_try, H, N_FLOORS, S22["floor_heights_m"])
    win_t = fb_try["patterns"][-1]["window"]
    print(f"입면 폭 {W_try:5.1f} m → 베이 {fb_try['bay_count']}개 × {W_try / fb_try['bay_count']:.2f} m, 창 폭 {win_t['w_m']:.2f} m")

## 8. BDS → 3D — 결정론적 절차 모델러 (trimesh)

원본은 Blender `bmesh` 로 1,500줄이다. 여기서는 핵심만 150줄로 옮긴다.

* 입면마다 **입면 좌표**를 잡는다: `a` = 왼쪽 끝에서 가로 거리, `d` = 벽면에서 바깥(+)·안(−), `z` = 높이.
* **벽을 구멍 뚫지 않는다.** 창 아래·창 위·창 사이를 따로 상자로 쌓는다(불리언 연산 없음 — 빠르고 결과가 늘 같다).
* 창은 실제 깊이(리빌)만큼 **안으로 들어가 있고**, 창틀·멀리언·창대는 실제 두께를 가진 부재다.
* 커튼월은 층마다 불투명 띠(스팬드럴) + 비전 유리 + 멀리언.
* 1층 점포: 굽(하부 벽) + 유리 + 상부 간판 띠 + 문. 간판은 판 + 글자 띠.
* 옥상: 슬래브 + 파라펫 + 4번 노트북의 옥상 부재(주축 각도로 돌려 놓는다).

In [ ]:
WALL_T = 0.30
ROLE_PBR = {"wall": (0.85, 0.0), "glass": (0.08, 0.35), "frame": (0.35, 0.6), "spandrel": (0.3, 0.45),
            "sign": (0.5, 0.0), "letter": (0.4, 0.0), "roof": (0.95, 0.0), "core": (0.9, 0.0), "equip": (0.5, 0.4)}

def M(ps, role, color):
    name = f"{role}{color}"
    if name not in ps.mats:
        ps.material(name, color, *ROLE_PBR.get(role, (0.7, 0.0)))
    return name

def darker(hx, k=0.7):
    r, g, b = hex_rgb(hx)
    return "#{:02X}{:02X}{:02X}".format(int(r * k * 255), int(g * k * 255), int(b * k * 255))

class Kit:
    """입면 좌표 (a 가로, d 깊이 +바깥, z 높이) 로 상자를 놓는다."""
    def __init__(self, ps, geo):
        self.ps, self.W = ps, geo["width_m"]
        self.S = np.array([*geo["start"], 0.0])
        self.u = (np.array([*geo["end"], 0.0]) - self.S) / self.W
        self.n = np.array([*geo["normal"], 0.0])
    def box(self, group, mat, a0, a1, d0, d1, z0, z1):
        if a1 - a0 < 1e-3 or d1 - d0 < 1e-4 or z1 - z0 < 1e-3:
            return
        o = self.S + self.u * a0 + self.n * d0 + np.array([0, 0, z0])
        V, F = box_vf(o, self.u * (a1 - a0), self.n * (d1 - d0), [0, 0, z1 - z0])
        self.ps.add(group, mat, V, F)

def pattern_for(fb, k):
    return next((p for p in fb["patterns"] if k in p["floors"]), fb["patterns"][-1])

def window_cell(K, g, win, a0, a1, z0, z1, wallm):
    """개구부 하나: 리빌만큼 들어간 유리 + 창틀 + 멀리언 + 창대."""
    rec, fw = max(0.03, win.get("recess_m", 0.1)), win.get("frame_w_m", 0.06)
    glass, frame = M(K.ps, "glass", win["glass_color"]), M(K.ps, "frame", win["frame_color"])
    K.box(g, glass, a0, a1, -rec - 0.02, -rec, z0, z1)
    for b in [(a0, a0 + fw, z0, z1), (a1 - fw, a1, z0, z1), (a0, a1, z0, z0 + fw), (a0, a1, z1 - fw, z1)]:
        K.box(g, frame, b[0], b[1], -rec, -rec + 0.05, b[2], b[3])
    cols, rows = win.get("mullions", [2, 1])
    for c in range(1, cols):
        x = a0 + (a1 - a0) * c / cols
        K.box(g, frame, x - fw / 2, x + fw / 2, -rec, -rec + 0.04, z0, z1)
    for r_ in range(1, rows):
        z = z0 + (z1 - z0) * r_ / rows
        K.box(g, frame, a0, a1, -rec, -rec + 0.04, z - fw / 2, z + fw / 2)
    K.box(g, wallm, a0 - 0.05, a1 + 0.05, -rec, 0.06, z0 - 0.06, z0)        # 창대 (밖으로 6 cm)

def punched_floor(K, g, bays, win, z0, z1, wallm):
    fh = z1 - z0
    sill = min(win["sill_m"], fh - 0.6)
    hh = min(win["h_m"], fh - sill - 0.2)
    for b0, b1 in bays:
        bw = b1 - b0
        if win["kind"] == "ribbon":
            opens = [(b0 + 0.35, b1 - 0.35)]
        else:
            n = win["per_bay"]
            ww = min(win["w_m"], bw / n - 0.25)
            opens = [(b0 + (j + .5) * bw / n - ww / 2, b0 + (j + .5) * bw / n + ww / 2) for j in range(n)]
        K.box(g, wallm, b0, b1, -WALL_T, 0, z0, z0 + sill)                    # 창 아래
        K.box(g, wallm, b0, b1, -WALL_T, 0, z0 + sill + hh, z1)               # 창 위
        edges = [b0] + [x for o in opens for x in o] + [b1]
        for i in range(0, len(edges), 2):                                     # 창 사이 기둥
            K.box(g, wallm, edges[i], edges[i + 1], -WALL_T, 0, z0 + sill, z0 + sill + hh)
        for a0, a1 in opens:
            window_cell(K, g, win, a0, a1, z0 + sill, z0 + sill + hh, wallm)

def curtain_floor(K, g, bays, win, z0, z1, spandrel_color):
    fh = z1 - z0
    vision = min(win["h_m"], fh - 0.3)
    glass, frame = M(K.ps, "glass", win["glass_color"]), M(K.ps, "frame", win["frame_color"])
    span = M(K.ps, "spandrel", spandrel_color)
    K.box(g, span, 0, K.W, -0.08, 0, z0, z1 - vision)                          # 불투명 띠
    K.box(g, glass, 0, K.W, -0.10, -0.06, z1 - vision, z1)                    # 비전 유리
    K.box(g, frame, 0, K.W, -0.06, 0.08, z0 - 0.04, z0 + 0.04)                # 층 경계 트랜섬
    cols = max(1, win.get("mullions", [1, 1])[0])
    for b0, b1 in bays:
        K.box(g, frame, b0 - 0.05, b0 + 0.05, -0.06, 0.12, z0, z1)            # 베이 멀리언
        for c in range(1, cols):
            x = b0 + (b1 - b0) * c / cols
            K.box(g, frame, x - 0.03, x + 0.03, -0.06, 0.06, z1 - vision, z1)
    K.box(g, frame, K.W - 0.05, K.W + 0.05, -0.06, 0.12, z0, z1)

def storefront_floor(K, g, units, z0, z1, wallm):
    fh, last = z1 - z0, 0.0
    for u in sorted(units, key=lambda u: u["x0_m"]):
        K.box(g, wallm, last, u["x0_m"], -WALL_T, 0, z0, z1)                   # 점포 사이 벽
        a0, a1 = u["x0_m"], u["x1_m"]
        bh, hd = u["bulkhead_m"], u["header_m"]
        K.box(g, M(K.ps, "wall", "#6E6A63"), a0, a1, -0.25, 0.02, z0, z0 + bh)
        K.box(g, M(K.ps, "glass", "#6F8FA3"), a0, a1, -0.22, -0.18, z0 + bh, z1 - hd)
        K.box(g, M(K.ps, "sign", "#2E3236"), a0, a1, -0.1, 0.12, z1 - hd, z1)   # 상부 간판 띠
        if u.get("door_x_m") is not None:
            d = u["door_x_m"]
            K.box(g, M(K.ps, "frame", "#3A3D40"), d - 0.55, d + 0.55, -0.20, -0.14, z0, z0 + 2.3)
        if u.get("awning"):
            K.box(g, M(K.ps, "sign", u["awning"]["color"]), a0, a1, 0, u["awning"]["depth_m"], z1 - hd - 0.35, z1 - hd)
        last = a1
    K.box(g, wallm, last, K.W, -WALL_T, 0, z0, z1)

def build_facade(ps, geo, fb, spans):
    K = Kit(ps, geo)
    fid = geo["id"]
    top = spans[-1][2]
    if fb["role"] in ("party_wall", "hidden") or not fb["patterns"]:
        K.box(f"{fid}_WALL", M(ps, "wall", fb["wall"]["color"]), 0, K.W, -WALL_T, 0, 0, top)
        return
    bays = fb.get("bays") or [[i * K.W / fb["bay_count"], (i + 1) * K.W / fb["bay_count"]] for i in range(fb["bay_count"])]
    for k, z0, z1 in spans:
        p = pattern_for(fb, k)
        win = p["window"]
        wc = (p.get("wall") or fb["wall"])["color"]
        wallm = M(ps, "wall", wc)
        g = f"{fid}_F{k:02d}"
        if k == 1 and fb.get("storefront"):
            storefront_floor(K, g, fb["storefront"], z0, z1, wallm)
        elif win["kind"] == "curtain_wall":
            curtain_floor(K, g, bays, win, z0, z1, win.get("spandrel_color") or (wc if (p.get("wall") or {}).get("class") == "curtain_wall" else darker(win["glass_color"])))
        elif win["kind"] == "none":
            K.box(g, wallm, 0, K.W, -WALL_T, 0, z0, z1)
        else:
            punched_floor(K, g, bays, win, z0, z1, wallm)
        if p.get("band"):
            b = p["band"]
            K.box(g, M(ps, "wall", b["color"]), 0, K.W, -0.02, b.get("proj_m", 0.05), z1 - b["h_m"], z1)
    for i, s in enumerate(fb.get("signs", [])):                                # 간판: 판 + 글자 띠
        g = f"{fid}_SIGN{i}"
        z0, z1 = s["z_m"], s["z_m"] + s["h_m"]
        if s["kind"] not in ("channel_letters", "window_text"):
            K.box(g, M(ps, "sign", s["board_color"]), s["x0_m"], s["x1_m"], 0, 0.12, z0, z1)
        ins = 0.12 * (s["x1_m"] - s["x0_m"])
        K.box(g, M(ps, "letter", s["text_color"]), s["x0_m"] + ins, s["x1_m"] - ins, 0.12, 0.16,
              z0 + 0.25 * s["h_m"], z1 - 0.25 * s["h_m"])

def floor_spans(fh):
    z = np.concatenate([[0.0], np.cumsum(fh)])
    return [(i + 1, float(z[i]), float(z[i + 1])) for i in range(len(fh))]

def prism(ps, group, mat, ring, z0, z1):
    poly = Polygon(ring).buffer(0)
    m = trimesh.creation.extrude_polygon(poly, z1 - z0)
    ps.add(group, mat, m.vertices + [0, 0, z0], m.faces)

ROOF_COL = {"stair_core": "#D9DCDD", "penthouse": "#D9DCDD", "elevator_machine": "#C9CCCD", "machine_room": "#C9CCCD",
            "water_tank": "#E6E6E2", "ac_cluster": "#9DA3A6", "condenser": "#8E9497", "chiller": "#8E9497",
            "duct": "#B8BEC2", "solar": "#1F2A44", "greenery": "#5E7A4E"}

def build_roof(ps, roof, items, axis_deg, top):
    prism(ps, "ROOF_SLAB", M(ps, "roof", "#7C807F"), FP, top - 0.3, top)
    ph = roof.get("parapet_h_m", 1.0)
    for geo in geom["facades"]:
        K = Kit(ps, geo)
        K.box("ROOF_PARAPET", M(ps, "wall", roof.get("parapet_color", "#C9C4BA")), 0, K.W, -0.25, 0, top, top + ph)
        K.box("ROOF_PARAPET", M(ps, "wall", roof.get("coping_color", "#B4B0A8")), -0.03, K.W + 0.03, -0.28, 0.03, top + ph, top + ph + 0.06)
    a = math.radians(axis_deg)
    u, v = np.array([math.cos(a), math.sin(a)]), np.array([-math.sin(a), math.cos(a)])
    su, sv = FP @ u, FP @ v
    org, ow, od = su.min() * u + sv.min() * v, su.max() - su.min(), sv.max() - sv.min()
    for it in items:
        step = (it["w_m"] if it["run_axis"] == "x" else it["d_m"]) + 0.4
        for k in range(it["count"]):
            c = org + it["x_m"] * ow * u + it["y_m"] * od * v + (k * step) * (u if it["run_axis"] == "x" else v)
            o = np.array([*(c - it["w_m"] / 2 * u - it["d_m"] / 2 * v), top])
            V, F = box_vf(o, [*(u * it["w_m"]), 0], [*(v * it["d_m"]), 0], [0, 0, it["h_m"]])
            ps.add(f"ROOF_{it['kind']}", M(ps, "equip", ROOF_COL.get(it["kind"], "#A0A4A6")), V, F)

def build_building(bds, roof_items, axis_deg):
    ps = PartSet()
    spans = floor_spans(bds["floor_heights_m"])
    top = spans[-1][2]
    core = np.array(Polygon(FP).buffer(-0.35, join_style=2).exterior.coords)[:-1]
    prism(ps, "CORE", M(ps, "core", "#3B4045"), core, 0, top)                  # 유리 뒤 내부 매스
    for geo in geom["facades"]:
        fb = bds["facades"].get(geo["id"])
        if fb:
            build_facade(ps, geo, fb, spans)
    build_roof(ps, bds["roof"], roof_items, axis_deg, top)
    return ps
print("모델러 준비 완료")

## 9. 판독한 입면을 끼워 넣고 짓는다

나머지 입면은 실제 run 의 BDS(`s22_bds.json`)를 그대로 쓰고, 6절에서 읽은 입면만 갈아 끼운다.
옥상은 4번 노트북의 `out/roof_items.json` 이 있으면 그것을, 없으면 실제 run 의 옥상 판독을 쓴다.

In [ ]:
BDS = copy.deepcopy(S22)
BDS["facades"][FACADE_ID] = FACADE_M
BDS["floor_heights_m"] = FH

rf = OUT / "roof_items.json"
if rf.exists():
    rj = json.loads(rf.read_text(encoding="utf-8"))
    ROOF_ITEMS, AXIS, ROOF_SRC = rj["items"], rj["axis_deg"], "4번 노트북 out/roof_items.json — " + rj["meta"]["source"]
else:
    full = json.loads((DATA / "bds.json").read_text(encoding="utf-8"))
    ROOF_ITEMS = [{k: it[k] for k in ("kind", "x_m", "y_m", "run_axis", "w_m", "d_m", "h_m", "count")} for it in full["roof"]["items"]]
    AXIS, ROOF_SRC = full["roof"]["axis_deg"], "실제 run 의 옥상 판독 (bds.json)"

t0 = time.time()
ps = build_building(BDS, ROOF_ITEMS, AXIS)
GLB = ps.glb()
st = ps.stats()
lo, hi = ps.bounds()
print(f"생성 {time.time() - t0:.1f}초 · 부재 {st['parts']:,}개 · 삼각형 {st['triangles']:,}개 · 재질 {st['materials']}종 · 그룹 {st['groups']}개")
print(f"크기 {hi[0] - lo[0]:.1f} × {hi[1] - lo[1]:.1f} × {hi[2] - lo[2]:.1f} m (대장 높이 {H:.1f} m) · glb {len(GLB) / 1e6:.2f} MB")
print("옥상:", ROOF_SRC)
(OUT / "ipark_bds.glb").write_bytes(GLB)

img = render(ps, size=(1280, 720), azim=200, elev=10)
Image.fromarray(img).save(OUT / "ipark_bds.png")
plt.figure(figsize=(16, 9)); plt.imshow(img); plt.axis("off")
plt.title(f"BDS → 결정론적 3D ({FACADE_ID} 판독: {READ_META['source']})", fontsize=15); plt.show()

In [ ]:
show_glb(GLB, note="마우스로 돌려 볼 수 있다. 인터넷이 없으면 빈 칸 — 위 정적 렌더를 볼 것.")

## 10. 같은 JSON → 같은 메시

이것이 v0.3 의 핵심 약속이다. 이미지 생성 모델은 같은 프롬프트에도 매번 다른 그림을 낸다.
여기서는 **같은 BDS 를 두 번 지어 파일의 지문(SHA-256)을 비교**한다. 그리고 숫자 하나(베이 수)만 바꿔 본다.

In [ ]:
glb_a = build_building(copy.deepcopy(BDS), ROOF_ITEMS, AXIS).glb()
glb_b = build_building(copy.deepcopy(BDS), ROOF_ITEMS, AXIS).glb()
print(f"같은 BDS 1회차  {sha(glb_a)}  ({len(glb_a):,} bytes)")
print(f"같은 BDS 2회차  {sha(glb_b)}  ({len(glb_b):,} bytes)  → 동일: {glb_a == glb_b}")

r2 = copy.deepcopy(READING)
r2["bay_count"] = int(r2["bay_count"]) + 1
fb2, fh2, _ = apply_reading(r2, KNOWN["width_m"], H, N_FLOORS, S22["floor_heights_m"])
bds2 = copy.deepcopy(BDS); bds2["facades"][FACADE_ID] = fb2
ps2 = build_building(bds2, ROOF_ITEMS, AXIS)
glb_c = ps2.glb()
print(f"베이 {READING['bay_count']} → {r2['bay_count']}       {sha(glb_c)}  ({len(glb_c):,} bytes)  → 부재 {ps.stats()['parts']:,} → {ps2.stats()['parts']:,}")

## 11. 저장

In [ ]:
(OUT / "facade_reading.json").write_text(json.dumps({"facade": FACADE_ID, "known": KNOWN, "meta": READ_META,
    "photo_credit": CREDIT, "reading": READING}, ensure_ascii=False, indent=1), encoding="utf-8")
(OUT / "bds_mini.json").write_text(json.dumps(BDS, ensure_ascii=False, indent=1), encoding="utf-8")

print("── 요약 ─────────────────────────────")
print(f"사진       {CREDIT['title']} ({CREDIT['author']}, {CREDIT['licence']})")
print(f"판독       {READ_META['source']} · 신뢰도 {READING['confidence']:.2f} · 베이 {READING['bay_count']} · 창 {READING['window']['kind']}")
print(f"환산       {notes[-1]}")
print(f"3D         부재 {st['parts']:,} · 삼각형 {st['triangles']:,} · 높이 {hi[2] - lo[2]:.1f} m · 지문 {sha(GLB)}")
for f in ["facade_reading.json", "bds_mini.json", "ipark_bds.glb", "ipark_bds.png"]:
    print("  out/" + f, f"{(OUT / f).stat().st_size / 1024:.0f} KB")

## 해 볼 것

1. 자기 동네 저층 상가 건물을 정면에서 찍어 올리고, 건축물대장의 층수·높이와 지도에서 잰 입면 폭을 `KNOWN` 에 넣어 보자.
   1층 점포(`storefront`)와 간판 글자가 제대로 읽히는가?
2. 같은 사진을 두 번 판독시키면 비율이 얼마나 흔들리나? 그 흔들림은 3D 에서 몇 cm 인가 (7절 표로 계산).
3. `need_shots` 에 무엇을 요청했나? 그 구도로 한 장 더 찍어 함께 넣으면(`content` 에 이미지 추가) 무엇이 달라지나.
4. 8절 모델러에 발코니를 추가해 보자. 스키마에 어떤 칸이 필요하고, 그 칸은 무엇에 곱해야 하나?
5. 이 방식이 **표현하지 못하는 건물**은? (곡면 저층부, 비정형 매스, 거대한 원형 조형…) → 6번 노트북.

### 참고
* addr2blend `docs/pipeline_api_reference.md` — "로드뷰를 3D 로 변환하는 API 는 없다"
* OpenAI Responses API — Structured Outputs (`json_schema`, `strict: true`)
* 예제 사진: *Yeongdong-daero 3* — kallerna, Wikimedia Commons, CC BY-SA 4.0